#Trabalho KNN
  - Nomes:
      - [João Henrique da Luz](https://github.com/Joao-H-Luz)
      - [Ighor](https://github.com/ighortelles)
      - [Frederico Fragoso Franke](https://github.com/f.franke)


# Bibliotecas
---


In [ ]:
import pandas as pd
from sklearn import neighbors
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

## Carregar o DF
---

In [ ]:
df = pd.read_csv("cardio_v2.csv", sep=";", encoding="latin1")
df.shape

(70000, 13)

In [ ]:
df


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110.0,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140.0,-1,3,1,0,0,1,1
2,2,18857,1,165,64.0,130.0,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150.0,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100.0,60,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,19240,2,168,76.0,120.0,80,1,1,1,0,1,0
69996,99995,22601,1,158,126.0,140.0,90,2,2,0,0,1,1
69997,99996,19066,2,183,105.0,180.0,90,3,1,0,1,0,1
69998,99998,22431,1,163,72.0,135.0,80,1,2,0,0,0,1


## Tratamento
---

In [ ]:
# Idade
df["age"] = df["age"].astype(str).str[-2:].astype(int)
df["age"] = 1900 + df["age"]
df.loc[df["age"] <= 1923, "age"] += 100

df["age"] = 2026 - df["age"]

# Altura
df["height"] = df["height"] / 100
df = df[df["height"] >= 1.32]

# IMC
df["imc"] = df.weight / (df.height / 100)**2

df

/tmp/ipykernel_2637/738262099.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["imc"] = df.weight / (df.height / 100)**2


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,imc
0,0,33,2,1.68,62.0,110.0,80,1,1,0,0,1,0,219671.201814
1,1,98,1,1.56,85.0,140.0,-1,3,1,0,0,1,1,349276.791584
2,2,69,1,1.65,64.0,130.0,70,3,1,0,0,0,1,235078.053260
3,3,3,2,1.69,82.0,150.0,100,1,1,0,0,1,1,287104.793250
4,4,52,1,1.56,56.0,100.0,60,1,1,0,0,0,0,230111.768573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,86,2,1.68,76.0,120.0,80,1,1,1,0,1,0,269274.376417
69996,99995,25,1,1.58,126.0,140.0,90,2,2,0,0,1,1,504726.806602
69997,99996,60,2,1.83,105.0,180.0,90,3,1,0,1,0,1,313535.787871
69998,99998,95,1,1.63,72.0,135.0,80,1,2,0,0,0,1,270992.510068


In [ ]:
df = df.dropna()
df.describe()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,imc
count,69894.000000,69894.000000,69894.000000,69894.000000,69894.000000,69894.000000,69894.000000,69894.000000,69894.000000,69894.000000,69894.000000,69894.000000,69894.000000,69894.000000
mean,49971.725756,52.410865,1.349529,1.644496,74.196937,128.826051,96.642001,1.367070,1.226543,0.088148,0.053781,0.803832,0.499728,274700.948750
std,28851.061641,28.793310,0.476825,0.078908,14.360589,154.125518,188.583052,0.680378,0.572363,0.283512,0.225588,0.397100,0.500004,52294.014220
min,0.000000,3.000000,1.000000,1.320000,10.000000,-150.000000,-70.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,34717.838657
25%,25009.250000,28.000000,1.000000,1.590000,65.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,238751.147842
50%,50004.500000,52.000000,1.000000,1.650000,72.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,263702.381435
75%,74884.000000,77.000000,2.000000,1.700000,82.000000,140.000000,90.000000,2.000000,1.000000,0.000000,0.000000,1.000000,1.000000,301194.019147
max,99999.000000,102.000000,2.000000,3.700000,200.000000,16020.000000,11000.000000,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000,857797.431936


## Separar a Base em Treino (80%) e Teste (20%)
---

In [ ]:
X = df.drop(columns=['cardio'])
X.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,imc
0,0,33,2,1.68,62.0,110.0,80,1,1,0,0,1,219671.201814
1,1,98,1,1.56,85.0,140.0,-1,3,1,0,0,1,349276.791584
2,2,69,1,1.65,64.0,130.0,70,3,1,0,0,0,235078.053260
3,3,3,2,1.69,82.0,150.0,100,1,1,0,0,1,287104.793250
4,4,52,1,1.56,56.0,100.0,60,1,1,0,0,0,230111.768573


In [ ]:
y = df["cardio"].values
print(y)

[0 1 1 ... 1 1 0]


In [ ]:
X_train, X_aux, y_train, y_aux = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

In [ ]:
X_validation, X_test, y_validation, y_test = train_test_split(
    X_aux, y_aux,
    test_size=0.5,
    random_state=42,
    stratify=y_aux)

## Treinamento
---

In [ ]:
# Modelo
clf = neighbors.KNeighborsClassifier(n_neighbors=5)

# Treino
clf.fit(X_train, y_train)

KNeighborsClassifier()

In [ ]:
# Teste
y_pred = clf.predict(X_test)
y_pred

array([0, 1, 0, ..., 0, 1, 1])

### Medidas
 ---

In [ ]:
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

Accuracy: 0.532618025751073


In [ ]:
acc = accuracy_score(y_validation, y_pred)
print("Accuracy:", acc)

ValueError: Found input variables with inconsistent numbers of samples: [6989, 6990]